<table>
  <tr>
    <td><div align="left"><font size="30">Robotics Toolbox for Python</font></div></td>
    <td><img src="https://raw.github.com/petercorke/robotics-toolbox-python/master/docs/figs/RobToolBox_RoundLogoB.png" width="300"></td>
  </tr>
</table>

<p></p>
<div align="center" style="font-size: 1.5em;">🤖🚀 Robotics without the cruft</div>
<p></p>


(c) Peter Corke 2026

In [ ]:
import importlib
import setup_tutorial
importlib.reload(setup_tutorial)
await setup_tutorial.setup_tutorial(required_toolboxes=['roboticstoolbox', 'pgraph'], required_packages=['matplotlib', 'numpy', 'scipy'])

print("\nrobotics toolbox version:", importlib.import_module("roboticstoolbox").__version__)

In [ ]:
# import matplotlib.pyplot as plt
import numpy as np
np.set_printoptions(linewidth=100, formatter={'float': lambda x: f"{x:8.3g}" if abs(x) > 1e-10 else f"{0:8.3g}"})

import math

# Mobile robotics

## Mobile robot kinematics

We can create a vehicle with bicycle kinematics, with a wheelbase of 2m.

In [ ]:
from roboticstoolbox import Bicycle

veh = Bicycle(L=2, workspace=10)


We can find the maximum path curvature it can achieve, which is the reciprocal of the minimum turning radius.

In [ ]:
print(veh.curvature_max)

The initial state of the vehicle $(x,y,\theta)$, where $\theta$ is the heading angle, can be set at contruction time but defaults to zero

In [ ]:
print(veh.x)

The state derivative $(\dot{x}, \dot{y}, \dot{\theta})$ is a function of current state and the inputs $(v, \gamma)$ where $\gamma$ is the angle of the steered wheel.

In [ ]:
xd = veh.deriv(u=(1, 0.2), x= [0,0,0])
print(xd)

which indicates motion in the x-direction and a change of heading angle.

The model supports simple animation of motion directed by a `control` function.  In this case the control sets a constant velocity and a steered wheel angle of 0.5 rad for $1<t<2$

In [ ]:
veh.run(10, control=lambda v, t, x: (1, 0.5 if 1<t<2 else 0), animate=True)

## Graph-based planning

In [ ]:
from pgraph import UGraph # from PGraph package
from roboticstoolbox import rtb_path_to_datafile # from Robotics Toolbox package
import json

# load an example route map from the rtb-data package
with open(rtb_path_to_datafile('data/queensland.json'), 'r') as f:
    data = json.loads(f.read())

g = UGraph() # create an undirected graph
for name, info in data['places'].items():
    g.add_vertex(name=name, coord=info["utm"]) # add places as vertices
for route in data['routes']:
    g.add_edge(route['start'], route['end'], cost=route['distance']) # add routes as edges

g.plot()

In [ ]:
path, length, parents = g.path_Astar('Hughenden', 'Brisbane')

g.plot(block=None)
g.highlight_path(path)

## Occupancy grid path planning

In [ ]:
house = rtb.rtb_load_matfile('data/house.mat')
floorplan = house['floorplan']
places = house['places']

pmarker = dict(markersize=6, color='y')
dx = rtb.DistanceTransformPlanner(floorplan, inflate=5)
dx.plan(places.kitchen)
dx.plot()

In [ ]:
p = dx.query(places.br3)

dx.plot(p, inflated=True, path_marker=pmarker)

## Sample-based planning

In [ ]:
house = rtb.rtb_load_matfile('data/house.mat')
floorplan = house['floorplan']
places = house['places']

prm = rtb.PRMPlanner(occgrid=floorplan, seed=0)
prm.plan(npoints=50)

prm.plot(edge=dict(alpha=0.3), vertex=dict(alpha=0.3))
print(prm)

In [ ]:
# reseed the PRN to get a workable solution
# prm = PRMPlanner(occgrid=floorplan, seed=2)
prm.plan(npoints=200)
prm.plot(edge=dict(alpha=0.1), vertex=dict(alpha=0.1))
print(prm)

## Planning with motion constraints

### Dubbins paths

In [ ]:
qs = (0, 0, math.pi/2)
qg = (1, 0, math.pi/2)

dubins = rtb.DubinsPlanner(curvature=1)
path, status = dubins.query(qs, qg)

dubins.plot(path)

### Reeds-Shepp path

In [ ]:
rs = rtb.ReedsSheppPlanner(curvature=1)
path, status = rs.query(qs, qg)

rs.plot(path)

### Configuration-space planning

Let's look at the classic piano mover's problem

In [ ]:
import math
import roboticstoolbox as rtb
import numpy as np

# start and goal configuration
qs = (2, 8, -math.pi/2)
qg = (8, 2, -math.pi/2)

# obstacle map
map = rtb.PolygonMap(workspace=[0, 10])
map.add([(5, 50), (5, 6), (6, 6), (6, 50)])
map.add([(5, 4), (5, -50), (6, -50), (6, 4)])

# create a polygon to represent the piano
length, width  =3, 1.5
piano = Polygon2(
    np.array([(-length / 2, width / 2), (-length / 2, -width / 2), (length / 2, -width / 2), (length / 2, width / 2)]).T
)

# the piano has bicycle kinematics with a maximum steering angle of 1 radian and a wheelbase of 2m (ok, it's an odd piano)
vehicle = rtb.Bicycle(steer_max=1, L=2, polygon=piano)

rrt = rtb.RRTPlanner(map=map, vehicle=vehicle, verbose=False, npoints=50, showsamples=True, seed=0)

We'll build an RRT with its goal `qg` in the lower-right of the figure

In [ ]:
map.plot()
rrt.plan(goal=qg)

We have found a bunch of piano poses that don't intersect with the red obstacle, a dot represents the position $(x,y)$ and the translucent rectangle indicates its orientation.  Then we joined them up using an RRT.

Now we can query for a path to the goal, given a start pose `qs` which is in the upper left of the figure.

In [ ]:
path, status = rrt.query(start=qs)
print(status)

A path has been found and is described by a set of RRT vertices with pose $(x,y,\theta)$.

In [ ]:
map.plot()
rrt.g.plot(colorcomponents=False, text=False, force2d=True,
    vopt=dict(color='darkblue', marker='o', markersize=10), 
    eopt=dict(color='darkblue', linewidth=3), block=None)
rrt.g.highlight_path(status.vertices, color='r')


The planner returns a smoothed path by fitting Dubins curves between the RRT vertices, resulting in a path that is driveable by the piano

In [ ]:
with np.printoptions(threshold=20):
    print(path)

We can overlay that on the RRT

In [ ]:
map.plot()
rrt.g.plot(colorcomponents=False, text=False, force2d=True,
    vopt=dict(color='darkblue', marker='o', markersize=10), 
    eopt=dict(color='darkblue', linewidth=3))
rrt.plot(path);

and animate it as a series of snapshots of the piano on its journey

In [ ]:
va = rtb.VehiclePolygon(piano)
map.plot(block=None)
for i in np.unique(np.rint(np.linspace(0, len(path) - 1, 20)).astype(int)):
    va.plot(path[i, :], alpha=0.2)


## Pose-graph optimization

In [ ]:
pg = rtb.PoseGraph('data/killian-small.toro')

pg.plot(text=False, block=None)


In [ ]:

pg.optimize()

pg.plot(text=False, block=None)


# Arm robotics

In [ ]:
from roboticstoolbox import models

panda = models.DH.Panda()

print(panda)

In [ ]:
q = panda.qr

panda.plot(q)

In [ ]:
panda.fkine(q)

In [ ]:
sol = panda.ikine_QP(SE3(0.5, 0.2, 0.5) * SE3.Rx(math.pi) )
print(sol)

In [ ]:
q = sol.q
panda.fkine(q)

In [ ]:
tg = jtraj(panda.qz, panda.qr, 100)
print(tg)

In [ ]:
panda.plot(tg.q)

In [ ]:
from roboticstoolbox import models

panda = models.ETS.Panda()
panda

In [ ]:
ets = panda.ets()
print(ets)

In [ ]:
from roboticstoolbox import Robot

r = Robot(ets)
print(r)

In [ ]:
panda.fkine(q)

In [ ]:
panda.jacob0(panda.qr)

In [ ]:
panda.manipulability(panda.qr)

In [ ]:
panda.jacobe(q)

In [ ]:
H = panda.hessian0(q)
H.shape

In [ ]:
panda.hasgeometry

In [ ]:
panda = models.URDF.Panda()
panda

In [ ]:
# panda.plot(q, backend="swift")

In [ ]:
panda.hasgeometry


In [ ]:
panda.hasdynamics

In [ ]:
puma = models.DH.Puma560()
print(puma)


In [ ]:
puma.dynamics()

In [ ]:
q = puma.qn

puma.inertia(q)

In [ ]:
puma.gravload(q)

In [ ]:
puma.coriolis(q, [1, 0, 0, 0,0, 0])

In [ ]:
puma.inertia_x(q)

In [ ]:
puma.plot(q, backend="pyplot")
puma.fellipse(q)

In [ ]:
puma.gravload_x(q)